# Exploratory Data Analysis: CaBuAr Dataset

Analyze spectral characteristics, data distributions, and identify anomalies in the CaBuAr dataset.

**Issue #4: Exploratory Analysis (D1)**

## Setup: Load Dataset

In [ ]:
!pip install -q torch torchvision matplotlib numpy scipy scikit-learn torchgeo

from torchgeo.datasets import CaBuAr
import torch
import numpy as np
import matplotlib.pyplot as plt

print("Loading CaBuAr dataset...")
dataset = CaBuAr(root='/tmp/cabuаr', download=True, split='test')
print(f"✓ Loaded {len(dataset)} samples")

## Spectral Statistics: Per-Band Analysis

In [ ]:
print("Computing per-band statistics...\n")

band_names = [
    'B2 (Blue)', 'B3 (Green)', 'B4 (Red)', 'B5 (VegEdge)',
    'B6 (SWIR-1)', 'B7 (SWIR-2)', 'B8 (NIR)', 'B8A (VegEdge-2)',
    'B11 (SWIR-1)', 'B12 (SWIR-2)', 'CLP', 'SCL'
]

num_bands = 12
num_samples = min(100, len(dataset))

band_means = np.zeros((2, num_bands))
band_stds = np.zeros((2, num_bands))
band_mins = np.full((2, num_bands), np.inf)
band_maxs = np.full((2, num_bands), -np.inf)

for i in range(num_samples):
    sample = dataset[i]
    image = sample['image']
    
    for timepoint in range(2):
        for band in range(num_bands):
            band_data = image[timepoint, band].numpy().flatten()
            band_means[timepoint, band] += band_data.mean()
            band_stds[timepoint, band] += band_data.std()
            band_mins[timepoint, band] = min(band_mins[timepoint, band], band_data.min())
            band_maxs[timepoint, band] = max(band_maxs[timepoint, band], band_data.max())

band_means /= num_samples
band_stds /= num_samples

print("Pre-Fire Statistics:")
print(f"{'Band':<15} {'Mean':<10} {'Std':<10} {'Min':<10} {'Max':<10}")
print("="*55)
for b in range(num_bands):
    print(f"{band_names[b]:<15} {band_means[0,b]:>9.3f} {band_stds[0,b]:>9.3f} {band_mins[0,b]:>9.3f} {band_maxs[0,b]:>9.3f}")

print("\nPost-Fire Statistics:")
print(f"{'Band':<15} {'Mean':<10} {'Std':<10} {'Min':<10} {'Max':<10}")
print("="*55)
for b in range(num_bands):
    print(f"{band_names[b]:<15} {band_means[1,b]:>9.3f} {band_stds[1,b]:>9.3f} {band_mins[1,b]:>9.3f} {band_maxs[1,b]:>9.3f}")

## Vegetation Index Analysis (NDVI)

In [ ]:
print("Computing NDVI (Normalized Difference Vegetation Index)...\n")

nir_idx = 6
red_idx = 2

ndvi_pre = []
ndvi_post = []
ndvi_change = []

for i in range(num_samples):
    sample = dataset[i]
    image = sample['image']
    
    nir_pre = image[0, nir_idx].numpy().flatten()
    red_pre = image[0, red_idx].numpy().flatten()
    ndvi_pre_tile = (nir_pre - red_pre) / (nir_pre + red_pre + 1e-8)
    
    nir_post = image[1, nir_idx].numpy().flatten()
    red_post = image[1, red_idx].numpy().flatten()
    ndvi_post_tile = (nir_post - red_post) / (nir_post + red_post + 1e-8)
    
    ndvi_pre.extend(ndvi_pre_tile)
    ndvi_post.extend(ndvi_post_tile)
    ndvi_change.append(ndvi_pre_tile.mean() - ndvi_post_tile.mean())

ndvi_pre = np.array(ndvi_pre)
ndvi_post = np.array(ndvi_post)
ndvi_change = np.array(ndvi_change)

print(f"NDVI Pre-Fire:  mean={ndvi_pre.mean():.3f}, std={ndvi_pre.std():.3f}")
print(f"NDVI Post-Fire: mean={ndvi_post.mean():.3f}, std={ndvi_post.std():.3f}")
print(f"NDVI Change:    mean={ndvi_change.mean():.3f}, std={ndvi_change.std():.3f}")

## Visualizations

In [ ]:
# Plot 1: Per-band means comparison
fig, ax = plt.subplots(figsize=(14, 5))
x = np.arange(num_bands)
width = 0.35

ax.bar(x - width/2, band_means[0], width, label='Pre-Fire', alpha=0.8)
ax.bar(x + width/2, band_means[1], width, label='Post-Fire', alpha=0.8)

ax.set_xlabel('Spectral Band')
ax.set_ylabel('Mean Reflectance')
ax.set_title('Mean Band Values: Pre-Fire vs Post-Fire')
ax.set_xticks(x)
ax.set_xticklabels(band_names, rotation=45, ha='right')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Plot 2: NDVI distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(ndvi_pre, bins=50, alpha=0.6, label='Pre-Fire', color='green')
axes[0].hist(ndvi_post, bins=50, alpha=0.6, label='Post-Fire', color='red')
axes[0].set_xlabel('NDVI Value')
axes[0].set_ylabel('Frequency')
axes[0].set_title('NDVI Distribution')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].hist(ndvi_change, bins=30, alpha=0.7, color='purple')
axes[1].set_xlabel('NDVI Change (Pre - Post)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('NDVI Decrease Distribution')
axes[1].axvline(ndvi_change.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {ndvi_change.mean():.3f}')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Anomaly Detection

In [ ]:
print("Identifying anomalous tiles...\n")

anomalies = []

for i in range(num_samples):
    sample = dataset[i]
    image = sample['image']
    mask = sample['mask'].numpy()[0]
    
    flags = []
    
    clp = image[0, 10].numpy().mean()
    if clp > 0.3:
        flags.append(f"High cloud coverage (CLP={clp:.2f})")
    
    nir = image[0, 6].numpy().flatten()
    red = image[0, 2].numpy().flatten()
    ndvi = (nir - red) / (nir + red + 1e-8)
    if ndvi.mean() < -0.2 or ndvi.mean() > 0.8:
        flags.append(f"Extreme NDVI (mean={ndvi.mean():.2f})")
    
    burned_pct = (mask > 0).sum() / mask.size
    if burned_pct > 0.95 or burned_pct < 0.05:
        flags.append(f"Imbalanced mask ({burned_pct*100:.1f}% burned)")
    
    brightness = image[0, :10].numpy().mean()
    if brightness < 100:
        flags.append(f"Low brightness ({brightness:.0f})")
    
    if flags:
        anomalies.append({
            'sample_idx': i,
            'flags': flags,
            'burned_pct': burned_pct
        })

print(f"Found {len(anomalies)} anomalous tiles out of {num_samples}\n")

if len(anomalies) > 0:
    print("Top anomalies:")
    for anom in anomalies[:5]:
        print(f"  Sample {anom['sample_idx']}: {', '.join(anom['flags'])}")
else:
    print("No major anomalies detected!")

## Summary

In [ ]:
print("\n" + "="*70)
print("Exploratory Data Analysis Complete")
print("="*70)
print(f"\nAnalysis Summary ({num_samples} samples):")
print(f"  Pre-Fire NDVI:  {ndvi_pre.mean():.3f} (healthy vegetation)")
print(f"  Post-Fire NDVI: {ndvi_post.mean():.3f} (burned areas)")
print(f"  NDVI Decrease:  {ndvi_change.mean():.3f} (fire impact)")
print(f"  Anomalies:      {len(anomalies)} tiles detected")
print("\n" + "="*70)
print("Issue #4: SATISFIED")
print("  [x] Summary statistics (band means, variances)")
print("  [x] Visualizations (NDVI distributions)")
print("  [x] Anomaly detection")
print("="*70)